---
title: 1a) neuPrint Tutorial - Querying Neuronal Connections with the neuPrint API
authors:
  - name: Aarushi Vardhan
    affiliations:
      - University of Toronto / University of Cambridge
  - name: Sapolnach Prompiengchai
    affiliations:
      - University of Oxford
---

## What Will You Learn in This Tutorial?

In this tutorial, we will begin using the **neuPrint API to explore a brain region involved in learning and memory: the mushroom body**. In particular, we will investigate a type of neuron, called **Kenyon cells (KCs)**.

Along the way, you will learn how to:

- use **Neuron Criteria** to select neurons of interest;
- retrieve neuronal connections using `fetch_adjacencies()`;
- distinguish **presynaptic (source)** and **postsynaptic (target)** neurons;
- work with **pandas DataFrames**;
- combine connectivity data with information about **cell types and neuron identities**;
- inspect and clean your results.

By the end, you will have created a dataset describing the neurons connected to Kenyon cells that we can use in the next tutorial to begin asking deeper questions about **how the mushroom body is organized and how information may flow through its circuits**.


In [ ]:
# 1. load your packages at the top
import pandas as pd
import plotly.express as px

# Load your personal info from the .env file made earlier.
# See the Programming Guide: 7_introducing_git.md for instructions.
import os
from dotenv import load_dotenv
from neuprint import Client
load_dotenv()
c = Client(os.getenv("NEUPRINT_SERVER"), dataset=os.getenv("NEUPRINT_DATASET"), token=os.getenv("NEUPRINT_TOKEN"))


# Make a folder to store your scripts / data. Eg:
project_dir = '/Users/aarushivardhan/Desktop/MB/'

For this tutorial, we will start by exploring a region of the fly brain called the **mushroom body (MB)**. Before we start exploring it with neuPrint, let's find out what makes this part of the brain so interesting.

## Why is it called the mushroom body?

Take a look at the image below. Can you see why scientists gave this structure its name?

<div style="text-align: center;">

:::{figure} ../static/mb.png
:width: 500px
**The *Drosophila* mushroom body.** When viewed under a microscope, the mushroom body has a shape that looks surprisingly similar to a mushroom! Scientists can make the structure easier to see using GFP (green fluorescent protein), which makes the neurons glow green. *Figure courtesy of Katrin Vogt.*
:::

</div>

The mushroom body is found in the brains of many invertebrates, including insects, spiders, and scorpions. In the fruit fly *Drosophila melanogaster*, the mushroom body contains roughly **2,500 neurons**, most of which are small neurons called **Kenyon cells**.

## What does the mushroom body look like?

Let's zoom in.

<div style="text-align: center;">

:::{figure} ../static/mb_zoom.png
:width: 500px

**Close-up of the *Drosophila* mushroom body**. Figure courtesy of Campbell and Turner, 2010.
:::

</div>

There are a few important parts to notice:

- **Calyx** — where Kenyon cells receive information from other neurons.
- **Stalk** — where the axons of Kenyon cells bundle together.
- **Lobes** — where Kenyon cells send information to other parts of the MB.

You can think of the mushroom body as a place where information comes in, gets processed, and is then sent out to other parts of the brain.

And this is where neuPrint becomes really useful.
 
> Instead of only looking at a picture of the mushroom body, we can use the fly's connectome to ask: **Who is connected to whom?**

## So, what does the mushroom body actually do?

The mushroom body is involved in **learning and memory**.

For example, researchers can train a fly to associate a particular smell with something unpleasant, such as an electric shock. After learning this association, the fly can change its behavior when it encounters that smell again.

Scientists have found that the mushroom body is essential for this type of learning. When researchers block the output of the mushroom body, flies have difficulty performing these learned behaviors.

This raises an interesting question:

> **How can a network of neurons help a fly learn?**

We can start investigating this question using the connectome.

## Uncovering How a Brain Learns

Imagine that a fly encounters a smell. Information about that smell travels through the brain and eventually reaches the **mushroom body**.

A simplified version of this pathway looks something like:

> **Smell → sensory neurons → mushroom body → downstream neurons → behavior**

As you can see, the mushroom body does not work alone. It receives information from other parts of the brain and sends information to neurons downstream. This gives us several questions that we can investigate using **neuPrint**.

### Let's Start Simple

We know that the mushroom body contains 2,500 neurons of which most of Kenyon cells. Do all of these Kenyon cells do the same thing? Or, are different Kenyon cells connected to different neurons? Using neuPrint, we can explore the connections of individual Kenyon cells and investigate where their information comes from and where it goes.

### Challenge: Meet a Kenyon Cell

Let's explore a Kenyon cell and its connections!

As you explore, consider the following questions:

1. **Which cell types send information to the Kenyon cell of interest?**
2. **Which cell types receive information from the Kenyon cell of interest?**
3. **Are some partners cell types more strongly connected to the Kenyon cell of interest than others?**
4. **Do different Kenyon cell types connect to similar cell types, or do they have distinct connectivity patterns?**
5. **Do different Kenyon cell types receive and send information with similar strengths?**

:::{important}
**Terminology: Cell Type**

Neurons belonging to the same cell type share a common biological identity and are therefore given the same type name. For a more intuitive example, refer to [What are cell types?](fly_brain_and_neuropils.md#5-from-individual-neurons-to-cell-types).
:::


> **Think like a scientist:**
> Don't worry about finding the "right" answer immediately. Start by exploring the connections and see what patterns you can find. Your observations may lead to new questions!

To begin exploring connectivity in the mushroom body (MB), we first need a way to retrieve connections between neurons and determine their connection strength, inferred from the connectome by the number of synapses.

## Fetching connections

The `fetch_adjacencies` function allows us to retrieve connections between neurons, including the number of synapses connecting them.

We will also use **Neuron Criteria** (`NC`) to fine-tune our queries. This allows us to specify exactly which neurons or brain regions we are interested in. In this case, we want to focus on the **mushroom body (MB)**, and more specifically, **Kenyon cells (KCs)**.

Throughout this tutorial, you will notice that we regularly refer to the **neuPrint documentation** and highlight relevant sections that we explain in more detail. For this particular section, please refer to the [neuPrint Python documentation](https://connectome-neuprint.github.io/neuprint-python/docs/notebooks/QueryTutorial.html), which provides a useful guide to querying and fetching connections between neurons.

Let's import the functions we will need:

In [2]:
# In a typical Python script / notebook, imports are usually placed at the top.
# We are importing these functions here so you can see where they come from as we introduce them.
from neuprint import fetch_adjacencies, NeuronCriteria as NC

Before constructing our query, we need to determine how Kenyon cells are labeled in the dataset. A useful resource for exploring cell types in the male *Drosophila* CNS is the [Cell Type Explorer](https://reiserlab.github.io/celltype-explorer-drosophila-male-cns/).

Click **“Browse all types”**. We can then use the **Region** filter to explore cell types associated with the mushroom body (MB). Next, search for **Kenyon Cells** under **Class** in the Cell Type Explorer. You will see that Kenyon cell types have names beginning with **KC**.

> **Note:** If you are exploring a different cell type or brain region, you can simply type its name into the search bar. This example is mainly intended to show you how cell type names and abbreviations are specified in connectome nomenclature.

## Creating our Neuron Criteria

To search for specific neurons in neuPrint, we can use [**Neuron Criteria**](https://connectome-neuprint.github.io/neuprint-python/docs/notebooks/QueryTutorial.html#Neuron-Search-Criteria). The neuPrint documentation describes this as a way to specify neurons of interest by `bodyId`, `type`, `instance`, or a `NeuronCriteria` object.

Before using these criteria, it is helpful to understand what these terms mean. The [NeuPrint Data Model Terms](https://neuprint.janelia.org/public/neuprintuserguide.pdf) section of the neuPrint User Guide provides definitions for these concepts.

`bodyId`

A `bodyId` is the unique identifier assigned to an individual neuron in the connectome. For example, if we wanted to query one specific neuron, we could use its `bodyId`.

`type`

A `type` is the name assigned by biologists to describe a **cell type**. For example, Kenyon cell types have names beginning with `KC`.

`instance`

An `instance` provides additional information about an individual neuron, including the **hemisphere in which its cell body is located**.

When this information is known, `_L` and `_R` are appended to the cell type name to indicate the **left** and **right** sides of the brain, respectively.

For example:

- `KCa'b'-ap1` → the **cell type**
- `KCa'b'-ap1_L` → an **instance** of this cell type on the **left** side of the brain
- `KCa'b'-ap1_R` → an **instance** of this cell type on the **right** side of the brain

<div style="text-align: center;">

:::{figure} ../static/KCapbp-ap1_R.png
:width: 500px

**KCa'b'-ap1_R.** The cell body is circled in red, indicating that it is on the right side.
:::

:::{figure} ../static/KCapbp-ap1_L.png
:width: 500px

**KCa'b'-ap1_L.** The cell body is circled in red, indicating that it is on the left side.
:::

</div>

Thus, the `type` tells us **what kind of neuron it is**, while the `instance` provides additional information that distinguishes an individual neuron or group of neurons, such as the side of the brain where its cell body is located.

For our analysis, we are interested in **Kenyon cells**. Rather than specifying individual `bodyId`s, we can use the `type` field to search for Kenyon cell types. Since Kenyon cell types begin with `KC`, we can use a regular expression to select them.

We will use `*` which acts as a wildcard, meaning that this criterion will match any celltype beginning with `KC`. For example, this can match cell types such as `KCg`, `KCab`, or other Kenyon cell subtypes whose names begin with `KC`.

:::{important}
**Be careful with regular expressions!**

Wildcards can sometimes match more cell types than you intended. For example, a query such as `Mi1*` may match `Mi1`, `Mi11`, `Mi12`, and other names beginning with `Mi1`.

Always check the output of your query to make sure that you retrieved exactly the neurons you intended. This is an iterative process: construct your query, inspect the results, and refine it if necessary.
:::

If we look at the **Neuron Criteria** documentation, we can specify the `status` and `cropped` fields to select neurons that have been fully annotated and reconstructed. For our analysis, we will use `status="Traced"` and `cropped=False` to exclude cropped neurons.

In [3]:
criteria = NC(type="KC.*", status="Traced", cropped=False)

We now have a criterion that we can use in our neuPrint queries to restrict our analysis to Kenyon cells.

## Fetching Connections

Let's return to the neuPrint documentation and look at the [**Fetch Connections**](https://connectome-neuprint.github.io/neuprint-python/docs/notebooks/QueryTutorial.html#Fetch-connections) section. Here, we find:

> "Find synaptic connection strengths between one set of neurons and another using `fetch_adjacencies()`."

> "The 'source' and/or 'target' neurons are selected using `NeuronCriteria`. Additional parameters allow you to filter by connection strength or ROI. Two DataFrames are returned, for neuron properties and per-ROI connection strengths."

Let's break this down.

**Source and Target Neurons**

A connection between two neurons can be represented as:

```text
Source neuron  →  Target neuron
````

The **source** neuron is the neuron sending the connection, while the **target** neuron is the neuron receiving the connection.

For example:

```text
Kenyon cell  →  downstream neuron
   source           target
```

We can use `NeuronCriteria` to specify which neurons should be included as the **source**, **target**, or both.

**Connection Strength**

`fetch_adjacencies()` also allows us to examine the **strength of connections**.

In neuPrint, connection strength is represented by the **number of synapses** connecting the source and target neurons.

For example:

```text
Neuron A  ───────→  Neuron B
             12 synapses
```

This tells us that there are **12 synapses** connecting Neuron A to Neuron B.

**Regions of Interest (ROIs)**

We can also restrict our search to specific **regions of interest (ROIs)**. This is useful when we only want to examine connections within a particular brain region. For example, we could restrict our analysis to the **mushroom body (MB)**.

For this tutorial, however, we will not restrict our query to a specific ROI. Instead, we are interested in the overall connections between Kenyon cells and their partners.

**What does `fetch_adjacencies()` return?**

When we run `fetch_adjacencies()`, it returns two DataFrames:

1. A DataFrame containing **neuron properties**.
2. A DataFrame containing **connection strengths for each ROI**.

We can use these DataFrames to identify the **neurons or cell types connected to our Kenyon cells** and examine the strength of those connections.

:::{important}

**What is a DataFrame?**

A **DataFrame** is a table-like data structure to store and organize data in rows and columns.

Each **row** represents an observation or entry, while each **column** represents a variable or property.

For example:

| neuron | type | bodyId |
| ------ | ---- | ------ |
| 1      | KCg  | 12345  |
| 2      | KCg  | 67890  |

Here, each row represents an individual neuron, while `neuron`, `type`, and `bodyId` are columns containing information about those neurons.
:::

In [4]:
# this is the formula for what you will put in:
# dataframe_1 , dataframe_2 = fetch_adjacencies(upstream_neuron_to_consider, downstream_neuron_to_considera)

# Fetch all input connections to KC neurons (or, neurons pre-synaptic / downstream to KCs)
input_neuron_df, input_conn_df = fetch_adjacencies(None, criteria)

# Fetch all output connections from KC neurons (or, neurons post-synaptic / downstream to KCs)
output_neuron_df, output_conn_df = fetch_adjacencies(criteria, None)

  0%|          | 0/21 [00:00<?, ?it/s]

  0%|          | 0/1 [00:00<?, ?it/s]

  0%|          | 0/21 [00:00<?, ?it/s]

  0%|          | 0/1 [00:00<?, ?it/s]

In [5]:
input_neuron_df

,bodyId,type,instance
0,11862,KCab-s,KCab-s_L
1,13173,KCab-c,KCab-c_R
2,14292,KCg-m,KCg-m_R
3,15103,KCg-m,KCg-m_R
4,17488,KCa'b'-ap2,KCa'b'-ap2_R
...,...,...,...
6189,903518148,NaN,NaN
6190,982607565,NaN,NaN
6191,985545304,NaN,KC_R_fragment
6192,1030770291,NaN,NaN


In [6]:
input_conn_df

,bodyId_pre,bodyId_post,roi,weight
0,10005,103144,gL(R),4
1,10009,52318,gL(L),1
2,10009,69855,gL(L),1
3,10009,80708,gL(L),1
4,10009,88000,gL(L),1
...,...,...,...,...
1017676,1035770569,74542,gL(R),1
1017677,1035770569,86183,CentralBrain-unspecified,1
1017678,1035770569,96304,gL(R),1
1017679,1035770569,144220,CentralBrain-unspecified,1


In input_neuron_df, you can see the `bodyId`, `type`, and `instance` columns of the neurons that **send input to Kenyon cells**.

As an exercise, try following the next steps with the **output DataFrame**. This will help you make sure you understand the workflow, since the examples
above focused specifically on the input connections. Remember to think carefully about the direction of the connection:

```text
Input to Kenyon cells:

Source neuron  →  Kenyon cell
   pre              post
```
For the output connections, the direction is reversed:
```text
Output from Kenyon cells:

Kenyon cell  →  Target neuron
    pre            post
```

In input_conn_df, you can see several important columns in our connection DataFrame:

- `bodyId_pre` indicates the neuron ID of the **pre-synaptic neuron**.
- `bodyId_post` indicates the neuron ID of the **post-synaptic neuron**.
- `roi` indicates the **brain region (ROI)** in which the synapses are located.
- `weight` indicates the **number of synapses** between the `bodyId_pre` and
  `bodyId_post` neurons.

However, we are not done yet. Remember our original goal:

1. **Which neurons send information into the mushroom body?**
2. **Which neurons receive information from the mushroom body?**

Our connection DataFrame tells us **which neurons are connected**, but it does
not yet tell us the **cell type identity** of those neurons.

We still need to determine the cell types of the neurons connecting to our
Kenyon cells, both for the neurons providing input to the KCs and the neurons
receiving output from the KCs.

## Merge DataFrames

To add this cell type information to our connection DataFrame, we can merge
our neuron properties with our connection data.

First, import the following function:

In [7]:
from neuprint import merge_neuron_properties

We now have two types of information:

- `input_neuron_df` tells us **what cell type each neuron belongs to**.
- `input_conn_df` tells us **which neurons are connected to one another**.

More specifically, `input_neuron_df` contains information linking each
`bodyId` to its corresponding `type` and `instance`.

`input_conn_df`, on the other hand, contains the connections between neurons.
It tells us the bodyId of the **presynaptic neuron** (bodyId_pre) and the
bodyId of the **postsynaptic neuron** (bodyId_post).

We therefore want to combine the information to reveal the **cell type** indentity of the bodyId_pre and bodyId_post
This is where [`merge_neuron_properties()`](https://connectome-neuprint.github.io/neuprint-python/docs/utils.html#neuprint.utils.merge_neuron_properties) is useful.

This takes a neuron DataFrame and a connection DataFrame and adds the requested
neuron properties to the connection table.

In [8]:
input_conn_df = merge_neuron_properties(input_neuron_df, input_conn_df, ['type', 'instance'])
input_conn_df

,bodyId_pre,bodyId_post,roi,weight,type_pre,instance_pre,type_post,instance_post
0,10005,103144,gL(R),4,AOTU019,AOTU019_R,KCg-m,KCg-m_R
1,10009,52318,gL(L),1,CT1,CT1_L,KCg-m,KCg-m_L
2,10009,69855,gL(L),1,CT1,CT1_L,KCg-m,KCg-m_L
3,10009,80708,gL(L),1,CT1,CT1_L,KCg-m,KCg-m_L
4,10009,88000,gL(L),1,CT1,CT1_L,KCg-m,KCg-m_L
...,...,...,...,...,...,...,...,...
1017676,1035770569,74542,gL(R),1,NaN,NaN,KCg-d,KCg-d_R
1017677,1035770569,86183,CentralBrain-unspecified,1,NaN,NaN,KCg-d,KCg-d_R
1017678,1035770569,96304,gL(R),1,NaN,NaN,KCg-d,KCg-d_R
1017679,1035770569,144220,CentralBrain-unspecified,1,NaN,NaN,KCg-d,KCg-d_R


Here:

- input_neuron_df provides the information about each bodyId.
- input_conn_df provides the connections between bodyId_pre and bodyId_post.
- properties=['type', 'instance'] tells neuPrint which neuron properties we want to add to the connection table.

The function performs the necessary merges and adds _pre and _post versions of each property.

So now, for each `bodyId_pre` and `bodyId_post`, we have the corresponding `instance` and `type` associated with that neuron.

Before continuing, let's perform a quick **sanity check** to make sure our query worked as intended.

Because we used Kenyon cells as our target criteria, we expect the post-synaptic neurons in our input DataFrame to be Kenyon cells.

We can check the `type_post` and  `instance_post` column to confirm this. Here, we are selecting the `type_post` column because we queried **connections into Kenyon cells.** 

Therefore, the Kenyon cells should be the **post-synaptic neurons**,
while the neurons making those input connections are the **pre-synaptic neurons.**

The .unique() method is a pandas method that returns the unique values found in a column.

In [9]:
input_conn_df['type_post'].unique()

<ArrowStringArray>
[     'KCg-m',     'KCg-s1', 'KCa'b'-ap2',      'KCg-d', 'KCa'b'-ap1',
     'KCg-s4',     'KCab-s',     'KCab-c',     'KCab-m',         'KC',
   'KCa'b'-m',     'KCab-p',     'KCg-s3',     'KCg-s2',        'KCg']
Length: 15, dtype: str

In [10]:
input_conn_df['instance_post'].unique()

<ArrowStringArray>
[      'KCg-m_R',       'KCg-m_L',      'KCg-s1_R',  'KCa'b'-ap2_R',
       'KCg-d_R',  'KCa'b'-ap1_R',      'KCg-s4_R',      'KCab-s_R',
      'KCab-c_R',      'KCab-m_R',          'KC_R',       'KCg-d_L',
  'KCa'b'-ap2_L',      'KCab-s_L',  'KCa'b'-ap1_L',    'KCa'b'-m_L',
      'KCab-m_L',      'KCab-c_L',      'KCab-p_L',      'KCab-p_R',
    'KCa'b'-m_R',      'KCg-s1_L',      'KCg-s3_L',      'KCg-s2_R',
      'KCg-s3_R',      'KCg-s2_L',      'KCg-s4_L', 'KC_L_fragment',
         'KCg_L']
Length: 29, dtype: str

We notice that some **fragment neurons** are still present in the `instance_post` columns. We can use this information to clean our DataFrame and remove these neurons from our analysis. We can also do the same for the `instance_pre` column.

We also notice that there are several missing values (`NaN`). Let's remove any rows where **at least one column contains a `NaN`**. This leaves us with connections where both the pre- and post-synaptic neurons are not labeled as fragments and all required columns contain values.


In [11]:
input_conn_df = input_conn_df[~input_conn_df["instance_post"].str.endswith("fragment", na=False)]
input_conn_df = input_conn_df[~input_conn_df["instance_pre"].str.endswith("fragment", na=False)]
input_conn_df = input_conn_df.dropna()

In [12]:
input_conn_df

,bodyId_pre,bodyId_post,roi,weight,type_pre,instance_pre,type_post,instance_post
0,10005,103144,gL(R),4,AOTU019,AOTU019_R,KCg-m,KCg-m_R
1,10009,52318,gL(L),1,CT1,CT1_L,KCg-m,KCg-m_L
2,10009,69855,gL(L),1,CT1,CT1_L,KCg-m,KCg-m_L
3,10009,80708,gL(L),1,CT1,CT1_L,KCg-m,KCg-m_L
4,10009,88000,gL(L),1,CT1,CT1_L,KCg-m,KCg-m_L
...,...,...,...,...,...,...,...,...
1016944,953362,580809,aL(L),2,KCab-m,KCab-m_L,KCab-m,KCab-m_L
1016945,953362,581393,bL(L),1,KCab-m,KCab-m_L,KCab-m,KCab-m_L
1016946,953362,915829,aL(L),1,KCab-m,KCab-m_L,KCab-m,KCab-m_L
1016947,953362,945591,bL(L),1,KCab-m,KCab-m_L,KCab-m,KCab-m_L


In [13]:
# Save the results to a dedicated project folder.
input_conn_df.to_csv(project_dir + "input_to_KC.csv", index=False)
